In [ ]:
# ── Run this chapter on a clean machine (Colab "Julia" runtime, Binder, or any Jupyter with a Julia kernel) ──
# Cell 1 of 2 — the engine and the data. Measured on a clean machine: about three minutes to a first fit.
# Both engines are public on GitHub, so nothing needs a registry.
import Pkg
Pkg.add(url = "https://github.com/itchyshin/DRM.jl", rev = "26f4c4ddca58bd672fce807a7564face3debdc93")   # the commit the chapters were executed against
Pkg.add(["DataFrames", "CSV", "Distributions", "StatsBase", "StatsModels"])
REPO_RAW = "https://raw.githubusercontent.com/itchyshin/stats-hours/main"   # works once the repository is public
for f in ("tools/theme_itchy.jl", "tools/figures.jl", "tools/engine-pin.txt", "data/2012/MBodySize.csv", "data/2012/BodySize.csv", "data/2012/ChickSurvival.csv", "data/2012/FemaleSuccess.csv", "data/2012/SparrowSurvival.csv")
    mkpath(dirname(f)); isfile(f) || download("$REPO_RAW/$f", f)
end
println("engine and data ready — run the next cell for the plotting stack (several minutes; read on meanwhile)")

In [ ]:
# Cell 2 of 2 — the plotting stack. This is the slow part on a bare machine (about eight minutes measured;
# Binder pays it once at image build, so there it is seconds). Every figure in the chapter needs it.
import Pkg
Pkg.add(["Makie", "CairoMakie", "AlgebraOfGraphics"])
using CairoMakie
println("plotting ready")

---
title: "Class 10: not even the groups are strangers"
book: Stats Hours with Itchy
chapter: 10
type: book-chapter
status: draft
created: 2026-09-07
engines: DRM.jl 0.7.1 at 26f4c4ddc (Julia 1.10.0) · drmTMB 0.7.0 (R 4.6.0)
tags: [book, julia, animal-model, relatedness, heritability, pedigree, DRM.jl]
deck: "A one-generation pedigree, a matrix of ones and halves and quarters, and a heritability that lost most of itself the moment somebody put the nest the chick grew up in into the model."
status_tag: Draft
status_note: "Every number and figure on this page was produced when the site was built."
provenance: "Every Julia cell in this chapter was run when the site was built and its output printed."
caveat: "The data are real: 1950 Lundy Island house-sparrow chick records with their sire and dam, of which the 1675 that carry a mass are used here, read from data/2012/SparrowSurvival.csv (provenance in data/2012/README.md). The simulated quantities are drawn from models fitted to that file, from stated seeds, in cells you can read."
footer_note: "Stats Hours with Itchy · Class 10 of twelve rungs, ten in v1, plus a coda · draft, all code run, 2026-09-12"
---

# Class 10: not even the groups are strangers

> **What this chapter is not.** It is not a course in quantitative genetics, and it is not the
> chapter on phylogenetic comparative methods or meta-analysis, which belong together in another
> book. This is the last rung of version 1's climb, and it frees one thing: the assumption that
> random effects are independent **of each other**.

---

## Objectives

By the end of this class you should be able to:

1. Say what a relatedness matrix is, fill it in for a one-generation pedigree, and check a few of its entries by hand.
2. Fit an animal model with `animal(1 | id)` and a supplied `A`, and read the additive genetic variance off the fit.
3. Compute a narrow-sense heritability as a ratio of variance components, with an interval, and say what the denominator contains.
4. Name the thing full sibs share that is not genes, fit it, and report what happens to h².
5. Decide, by simulation rather than by hope, whether your design can tell those two apart at all.
6. Recognise that a pedigree, a phylogeny and a map of sites are the same model with a different matrix in it.

---

## The class

**Itchy's office, 9:00 am. It is the last hard week, so JARO is here, with coffee and a printed pedigree. TOTO has brought a laptop and a hypothesis. MOMO has brought the data file. EDDIE has read ahead, and today it will cost him.**

**Itchy:** Class 6 said rows are not strangers, and gave every bird its own nudge. Class 7 let the nudges have slopes. Class 9 let them live inside a coin flip. Every one of those chapters made the same quiet promise, and today we break it. Momo, you have the file open. Read me the promise.

**Momo:** `u_j ~ N(0, σ_b²)`, independently for each *j*.

**Itchy:** Independently for each *j*. Every group drawn fresh, knowing nothing about any other group. That is a lie about sparrows, about species and about places, and today we replace it with a matrix.

In [ ]:
#| label: setup
using Random
# tools/diagnostics.jl includes tools/figures.jl, which includes
# tools/theme_itchy.jl itself, so one include does all three.
include("tools/diagnostics.jl")
using DRM, DataFrames, CSV, Statistics, LinearAlgebra, Printf, CairoMakie
set_theme!(theme_itchy(:light))

raw = CSV.read("data/2012/SparrowSurvival.csv", DataFrame; missingstring = ["NA", ""])
println("rows, columns: ", size(raw))
println(describe(raw, :nmissing, :eltype))
first(raw, 6)

**Toto:** There is a `Dad` column and a `Mum` column.

**Itchy:** There is, and that is the whole chapter. Every one of these chicks has a named sire and a named dam, so the file does not merely tell you which chicks are in a group; it tells you **how much** any two of them are related. Drop the rows with no mass and count what is left.

In [ ]:
#| label: clean
chicks = dropmissing(raw, [:Mass2])
chicks.Cohort = string.(chicks.Year)

n       = nrow(chicks)
n_sire  = length(unique(chicks.Dad))
n_dam   = length(unique(chicks.Mum))
n_brood = length(unique(chicks.BroodNo))
n_pair  = nrow(unique(chicks[!, [:Dad, :Mum]]))

mkpath("data/ch10")
CSV.write("data/ch10/chicks.csv", chicks)

@printf("chicks with a mass : %d  (of %d rows)\n", n, nrow(raw))
@printf("sires %d, dams %d, sire-dam pairs %d, broods %d\n", n_sire, n_dam, n_pair, n_brood)
@printf("cohorts            : %s\n", join(sort(unique(chicks.Year)), ", "))
println()
describe(chicks[!, [:Mass2]], :mean, :std, :min, :max)

**Itchy:** `{julia} n` chicks, `{julia} n_sire` fathers, `{julia} n_dam` mothers. Nobody is missing a parent, which is rarer than it sounds and is why we are using this file. Toto, what is the response?

**Toto:** `Mass2`. Chick mass, in grams, at the standard early weighing.

**Itchy:** Chick mass, and the question is the oldest question in the room: **how much of the difference between one chick and another is inherited?** Not "is there a difference" — how much. Count the sibships first, because the answer lives entirely in them.

In [ ]:
#| label: sibships
fam = combine(groupby(chicks, [:Dad, :Mum]), nrow => :k)
sort!(fam, :k, rev = true)

@printf("full-sib families (a sire-dam pair): %d\n", nrow(fam))
@printf("  largest %d chicks, median %d, %d singletons\n",
        maximum(fam.k), round(Int, median(fam.k)), count(==(1), fam.k))

sire_mates = combine(groupby(chicks, :Dad), :Mum => (m -> length(unique(m))) => :n)
dam_mates  = combine(groupby(chicks, :Mum), :Dad => (d -> length(unique(d))) => :n)
@printf("sires with more than one mate: %d of %d\n", count(>(1), sire_mates.n), nrow(sire_mates))
@printf("dams  with more than one mate: %d of %d\n", count(>(1), dam_mates.n),  nrow(dam_mates))

**Eddie:** So there are half sibs as well as full sibs.

**Itchy:** There are, and that matters more than the counts suggest. A design with nothing but full-sib families gives you one number — how alike siblings are — and asks you to believe that all of it is genetic. Half sibs give you a second number at a different relatedness, and two numbers at two relatednesses is the beginning of an argument rather than an assumption.

### The matrix that says who is related to whom

**Itchy:** Here is the object. It is called the **additive relationship matrix** and it is written *A*. One row and one column per individual. The entry *A*ᵢⱼ is twice the probability that a gene drawn at random from *i* and a gene drawn at random from *j* are copies of the same ancestral gene. Jaro, give them the recursion, because it is one line and it generates everything.

**Jaro:** *A*ᵢⱼ = ½(*A*ᵢ,ₛᵢᵣₑ₍ⱼ₎ + *A*ᵢ,dam₍ⱼ₎), and *A*ᵢᵢ = 1 + *F*ᵢ, with *F* the **inbreeding coefficient** — the chance that an individual's two copies of a gene are copies of one ancestral gene.

**Itchy:** One line, and every number falls out of it once you say what the founders are. Ours are the parents — the file gives us no parents of parents — so we assume they are unrelated and not inbred. Toto, turn the crank for two full sibs.

**Toto:** They have the same sire and the same dam, so it is a half of a half plus a half of a half.

**Itchy:** Which is a half. And two chicks who share a sire only?

**Toto:** A half of a half, plus a half of nothing. A quarter.

**Itchy:** *(writes on the board)*

> **With unrelated, non-inbred founders and one generation: A_ij = ¼·1[same sire] + ¼·1[same dam] for i ≠ j, and A_ii = 1. Full sibs ½, half sibs ¼, strangers 0.**

**Itchy:** That is not an approximation to the recursion, it is the recursion evaluated. Build it, and then — this is the part nobody does and everybody should — **check it**. The build is a comprehension with two `for`s in one set of brackets, `for i in 1:n, j in 1:n`, which makes a **matrix** rather than a vector. `sire[i] == sire[j]` is true or false, and a quarter times a truth is a quarter or nothing, which is the board's indicator. `offdiag` is a comprehension whose second `for` depends on the first, so it flattens into one vector of the entries above the diagonal; and `A'` is the transpose.

In [ ]:
#| label: build-A
# A_ij = 1/4 * [same sire] + 1/4 * [same dam] off the diagonal, 1 on it: the general
# recursion evaluated for ONE generation with unrelated, non-inbred founders.
sire, dam = chicks.Dad, chicks.Mum
A = [i == j ? 1.0 : 0.25 * (sire[i] == sire[j]) + 0.25 * (dam[i] == dam[j])
     for i in 1:n, j in 1:n]

offdiag = [A[i, j] for i in 1:n for j in (i + 1):n]
n_full  = count(==(0.5),  offdiag)
n_half  = count(==(0.25), offdiag)
n_none  = count(==(0.0),  offdiag)

@printf("A is %d x %d, symmetric: %s\n", size(A, 1), size(A, 2), string(A == A'))
@printf("pairs at 1/2 (full sibs)  : %d\n", n_full)
@printf("pairs at 1/4 (half sibs)  : %d\n", n_half)
@printf("pairs at 0   (unrelated)  : %d\n", n_none)
@printf("distinct off-diagonal values: %s\n", string(sort(unique(offdiag))))
@printf("smallest eigenvalue: %.4f  (a legal covariance, i.e. positive definite: %s)\n",
        minimum(eigvals(Symmetric(A))), string(isposdef(Symmetric(A))))

**Momo:** Three values and nothing else, which is what the board says there should be.

**Itchy:** Three values and nothing else. The last line checks that the matrix can be a covariance at all: a covariance may never say some combination of the chicks has *negative* variance, and a positive smallest eigenvalue rules that out. A matrix that is *shaped* right can still be *indexed* wrong, though, and no eigenvalue will tell you. So check entries against the file, by name. The hunt is `findfirst` with `any` inside it: the first chick for which *any* other chick is a full sib *and* any other is a half sib.

In [ ]:
#| label: check-A
# Find a chick that has all three kinds of relative in the file, then read A at
# each pair BY NAME.
full_sib(i, j) = j != i && sire[j] == sire[i] && dam[j] == dam[i]
half_sib(i, j) = j != i && (sire[j] == sire[i]) + (dam[j] == dam[i]) == 1   # exactly one parent shared
stranger(i, j) = sire[j] != sire[i] && dam[j] != dam[i]

i0 = findfirst(i -> any(j -> full_sib(i, j), 1:n) && any(j -> half_sib(i, j), 1:n), 1:n)
picks = (("self", i0),
         ("full sib",  findfirst(j -> full_sib(i0, j), 1:n)),
         ("half sib",  findfirst(j -> half_sib(i0, j), 1:n)),
         ("unrelated", findfirst(j -> stranger(i0, j), 1:n)))

for (label, j) in picks
    @printf("%-10s %s (sire %s, dam %s) vs %s (sire %s, dam %s) -> A = %.2f\n",
            label, chicks.ChickNo[i0], sire[i0], dam[i0],
            chicks.ChickNo[j], sire[j], dam[j], A[i0, j])
end

**Itchy:** Read the parent codes across each line and satisfy yourself. Same pair of parents, a half. One parent shared, a quarter. Nothing shared, nothing. Now look at it.

In [ ]:
#| label: fig-relatedness
#| fig-cap: "Heatmap of the additive relatedness matrix A, restricted to the chicks fathered by one busy sire, coloured from 0 to 1. The bright diagonal is each chick with itself, the blocks off it are full sibs sharing both parents at one-half, and the fainter wash everywhere else is half sibs by a different mother at one-quarter — there is no zero anywhere in this slice, because every chick shown shares the same father."
busy_sire = first(sort(sire_mates, :n, rev = true).Dad)
rows_shown = findall(==(busy_sire), sire)[1:min(end, 18)]
Ablock = A[rows_shown, rows_shown]

codes = chicks.ChickNo[rows_shown]
fig = Figure(size = (560, 480))
ax = Axis(fig[1, 1]; xlabel = "chick", ylabel = "chick",
    title = "relatedness among $(length(rows_shown)) chicks of one sire",
    xticks = (1:length(codes), codes), yticks = (1:length(codes), codes),
    xticklabelrotation = pi / 2, xticklabelsize = 8, yticklabelsize = 8,
    yreversed = true)
hm = heatmap!(ax, 1:length(rows_shown), 1:length(rows_shown), Ablock';
    colorrange = (0, 1))
Colorbar(fig[1, 2], hm; label = "A")
fig

**Eddie:** Blocks on the diagonal, and a faint wash everywhere else.

**Itchy:** The blocks are the offspring of one female by this male — full sibs, one half. The wash is every other chick he fathered with a *different* female — half sibs, one quarter. There is no zero anywhere, because every chick shown has the same father. That picture *is* the covariance the model is about to assume, and if it looks wrong to you now, it will look wrong in the estimate later and you will not know why.

### One term, and a matrix

**Itchy:** The model. It is Class 6's model with one word changed.

> **y = Xβ + a + ε, with a ~ N(0, σ_A² A) and ε ~ N(0, σ² I).**

**Itchy:** *a* is the vector of **breeding values**, one per chick: the part of a chick's mass that its genes are responsible for, summed over every locus. In Class 6 the random effect was `N(0, σ_b² I)` and the `I` was invisible because nobody writes it. Today it is an `A` and it is the whole content of the model. σ_A² is the **additive genetic variance**. Momo, before you see the code, what has actually changed?

**Momo:** Nothing about the shape. The random effect is still normal, still centred on zero, still has one variance. Only the levels are now correlated with each other, by a matrix I supplied rather than anything the model estimated.

**Itchy:** By a matrix you supplied, which is the sentence to keep. **The model does not learn who is related to whom; you tell it, and it estimates one number: how much that relatedness is worth.** In DRM.jl the term is `animal(1 | id)` — a **structured** random effect, one whose levels are correlated by a matrix you supply — and the matrix arrives as a keyword.

**Toto:** Class 8 spent an hour telling me that variance components want REML. So I am asking for REML.

**Itchy:** You may ask, and the engine will refuse: REML is not implemented for a structured random effect, so this model is fitted by maximum likelihood ([Where the engine stops](appendix-b-engine.html)). Class 8's argument still applies; it is the remedy that is unavailable. A maximum-likelihood variance component does not pay for the fixed effects estimated alongside it, so it comes out **too small**, and Class 8 told you the correction lands on whichever level has *fewer* units — not on the residual, where *n* against *n* − *p* is a factor of `{julia} round(n / (n - (1 + 1 + (length(unique(chicks.Year)) - 1))), digits = 4)` here and would reassure you for nothing. Before the hour is out we will measure what ML costs on this design, in a simulation where the answer is known. Now fit it.

In [ ]:
#| label: fit-animal
naive = drm(bf(@formula(Mass2 ~ Sex + Cohort + animal(1 | ChickNo)), @formula(sigma ~ 1)),
            Gaussian(); data = chicks, A = A)
println("converged: ", is_converged(naive))
naive

**Toto:** There is a `resd` line with `ChickNo` on it.

**Itchy:** That is σ_A on the log scale, and Class 6 taught you to read the heading before the number. Pull the pieces out and make the ratio.

In [ ]:
#| label: heritability-naive
sigma_A_naive = re_sd(naive)[:ChickNo]
sigma_e_naive = first(sigma(naive))
h2_naive = heritability(naive)

@printf("sigma_A (additive genetic) : %.4f g\n", sigma_A_naive)
@printf("sigma   (residual)         : %.4f g\n", sigma_e_naive)
@printf("V_A = %.4f    V_P = V_A + V_R = %.4f\n",
        sigma_A_naive^2, sigma_A_naive^2 + sigma_e_naive^2)
println()
@printf("h2 = V_A / V_P      : %.4f\n", h2_naive.estimate)
@printf("bias-corrected      : %.4f   (bias %+.4f, %+.1f%% of the ratio)\n",
        h2_naive.corrected, h2_naive.bias, 100 * h2_naive.bias / h2_naive.estimate)
@printf("delta-method SE     : %.4f\n", h2_naive.se)
@printf("95%% CI              : %.4f to %.4f\n", h2_naive.ci.lower, h2_naive.ci.upper)

**Itchy:** **h² = V_A / V_P**, additive genetic variance over **phenotypic variance** — the total variance in the trait, additive genetic plus everything else, V_A + V_R printed above — a ratio of variance components, exactly like Class 6's repeatability. "**Narrow-sense**" because only the *additive* slice of genetic variance sits in the numerator. Class 6 warned you that a ratio of estimates is not the estimate of a ratio, and said the correction would not be tiny here: it is `{julia} string(round(Int, 100 * h2_naive.bias / h2_naive.estimate), "%")` of the estimate, and the printed interval is centred on the corrected value. Report `corrected` and `bias` beside every ratio. Toto, say it as a sentence about sparrows.

**Toto:** About `{julia} string(round(Int, 100 * h2_naive.estimate), "%")` of the variation in chick mass is additive genetic, and the interval runs from `{julia} string(round(Int, 100 * h2_naive.ci.lower), "%")` to `{julia} string(round(Int, 100 * h2_naive.ci.upper), "%")`.

**Eddie:** That is a publishable number.

**Itchy:** It is a publishable number, and it is very probably wrong. Momo has had her hand up since the matrix went on the board.

### What else do full sibs share

**Momo:** They share a nest.

**Itchy:** They share a nest. Say the rest of it.

**Momo:** Every entry in that matrix says "these two chicks share half their genes". Not one entry says "these two chicks were fed by the same parents in the same nest on the same days in the same weather". But that is also true of them, and it would also make them alike, and the model has nowhere to put it — so it puts it in σ_A.

**Itchy:** That is the single most important paragraph in this chapter and Momo said it, not me. **A relatedness matrix is a hypothesis about why relatives resemble each other, and it is not the only one.** Full sibs share genes *and* a brood. If you fit only the genes, the brood has nowhere to go, and it goes into the estimate wearing the genes' name. Kruuk and Hadfield (2007) is a whole paper on exactly this.

**Toto:** But surely you cannot separate them. Full sibs are in the same nest by definition.

**Itchy:** In most datasets, that is exactly right, and the honest answer is "this design cannot tell them apart, so I will not pretend". In *this* file, look what the fieldwork did.

In [ ]:
#| label: crossing
per_brood = combine(groupby(chicks, :BroodNo),
                    :Dad => (x -> length(unique(x))) => :n_sire,
                    :Mum => (x -> length(unique(x))) => :n_dam,
                    nrow => :k)
mixed = count((per_brood.n_sire .> 1) .| (per_brood.n_dam .> 1))
both  = count((per_brood.n_sire .> 1) .& (per_brood.n_dam .> 1))

per_pair = combine(groupby(chicks, [:Dad, :Mum]),
                   :BroodNo => (x -> length(unique(x))) => :n_brood)
spread = count(>(1), per_pair.n_brood)

@printf("broods holding chicks of more than one sire-dam pair : %d of %d\n", mixed, n_brood)
@printf("  of those, broods where BOTH sire and dam differ    : %d\n", both)
@printf("sire-dam pairs whose chicks appear in >1 brood        : %d of %d\n", spread, n_pair)
@printf("brood sizes present: %s\n", string(sort(unique(per_brood.k))))

**Eddie:** A brood can contain chicks of two different pairs, and the two chicks differ in *both* parents, not just the father.

**Itchy:** In both parents, which is the tell. If this were extra-pair paternity — and these are house sparrows, so there is plenty of it — the sire would change and the dam would not, because a female lays her own eggs. Whole pairs changing means whole chicks were moved between nests. The file does not say why, and I will not call it a cross-fostering experiment when the file does not. The consequence is arithmetic, not archaeology: **`{julia} spread` sire–dam pairs have chicks in more than one brood, and `{julia} mixed` broods hold chicks of more than one pair, so "same parents" and "same nest" are different partitions of these `{julia} n` chicks.** Two different partitions can carry two different variance components. One partition cannot.

**Toto:** So we add the brood.

**Itchy:** We add the brood. It is an ordinary random intercept — no brood is more related to any other — which is a structured effect whose matrix is the identity, and beside `animal` the engine wants it written that way, as `relmat` with `K = I` ([Where the engine stops](appendix-b-engine.html)).

In [ ]:
#| label: fit-both
brood_levels = unique(chicks.BroodNo)
K_brood = Matrix{Float64}(I, length(brood_levels), length(brood_levels))

full_model = drm(bf(@formula(Mass2 ~ Sex + Cohort + animal(1 | ChickNo) + relmat(1 | BroodNo)),
                    @formula(sigma ~ 1)),
                 Gaussian(); data = chicks, A = A, K = K_brood,
                 algorithm = :sparse)     # the same fit, assembled sparsely: minutes become seconds
println("converged: ", is_converged(full_model))
full_model

**Momo:** `relmat(1 | BroodNo)` with an identity matrix is just `(1 | BroodNo)`.

**Itchy:** It is exactly `(1 | BroodNo)`, written the long way so you can see the family resemblance. **`animal`, `relmat`, `phylo` and `spatial` are one model with four sources for one matrix, and Class 6's `(1 | g)` is the same model with the boring matrix.** Now the numbers.

In [ ]:
#| label: partition
sA = re_sd(full_model)[:ChickNo]
sB = re_sd(full_model)[:BroodNo]
sE = first(sigma(full_model))

h2_full    = heritability(full_model; component = :ChickNo)
brood_full = heritability(full_model; component = :BroodNo)

V_A, V_B, V_R = sA^2, sB^2, sE^2
V_P = V_A + V_B + V_R

@printf("%-26s %8s %8s\n", "", "SD (g)", "share")
@printf("%-26s %8.4f %8.4f\n", "additive genetic (A)", sA, V_A / V_P)
@printf("%-26s %8.4f %8.4f\n", "brood",                sB, V_B / V_P)
@printf("%-26s %8.4f %8.4f\n", "residual",             sE, V_R / V_P)
@printf("%-26s %8.4f %8.4f\n", "phenotypic total",     sqrt(V_P), 1.0)
println()
@printf("h2 with brood     : %.4f  corrected %.4f (bias %+.1f%%)  SE %.4f  CI %.4f to %.4f\n",
        h2_full.estimate, h2_full.corrected, 100 * h2_full.bias / h2_full.estimate,
        h2_full.se, h2_full.ci.lower, h2_full.ci.upper)
@printf("brood share of V_P: %.4f  CI %.4f to %.4f\n",
        brood_full.estimate, brood_full.ci.lower, brood_full.ci.upper)
println()
# V_P is CONDITIONAL on the fixed effects: whatever Sex and Cohort explain sits outside it.
@printf("variance explained by Sex and Cohort : %.4f g^2  (outside V_P)\n",
        var(fitted(full_model)))
@printf("h2 on a FULL-variance denominator    : %.4f  (against %.4f above)\n",
        V_A / (V_P + var(fitted(full_model))), h2_full.estimate)
println()
# WHERE the brood variance came from. Not from the genetic block alone.
from_A = sigma_A_naive^2 - V_A
from_R = sigma_e_naive^2 - V_R
@printf("the brood block, %.4f g^2, was taken from\n", V_B)
@printf("  the genetic block : %.4f g^2   (%.1f%% of it)\n", from_A, 100 * from_A / V_B)
@printf("  the residual      : %.4f g^2   (%.1f%% of it)\n", from_R, 100 * from_R / V_B)
println()
# What each model believes about two full sibs.
@printf("covariance for a full-sib pair, pedigree-only model      : %.4f\n", 0.5 * sigma_A_naive^2)
@printf("two-component model, full sibs in the SAME brood         : %.4f\n", 0.5 * V_A + V_B)
@printf("two-component model, full sibs in DIFFERENT broods       : %.4f\n", 0.5 * V_A)

**Toto:** It went from `{julia} round(h2_naive.estimate, digits = 3)` to `{julia} round(h2_full.estimate, digits = 3)`.

**Itchy:** It lost about `{julia} string(round(Int, 100 * (1 - h2_full.estimate / h2_naive.estimate)), "%")` of itself, and the brood picked up `{julia} string(round(Int, 100 * brood_full.estimate), "%")` of the phenotypic variance — several times what is left over for genes. Eddie, you said the first number was publishable.

**Eddie:** I did.

**Itchy:** It is. That is the problem. Nothing about the first fit looked wrong: it converged, it printed an interval, and the interval was comfortably away from zero. The only thing wrong with it was a term that was not in it, and no diagnostic in this book can show you a term that is not in the model. Draw the two side by side.

In [ ]:
#| label: fig-partition
#| fig-cap: "Stacked bars of the same phenotypic variance split two ways: pedigree-only on the left, pedigree-plus-brood on the right. The two bars stand at nearly the same total height, so the brood block is not new variance — it is variance the pedigree-only model had nowhere to put, drawn mostly from the residual rather than from the genetic block."
labels = ["genetic (A)", "brood", "residual"]
naive_parts = [sigma_A_naive^2, 0.0, sigma_e_naive^2]
full_parts  = [V_A, V_B, V_R]

heights = vcat(naive_parts, full_parts)
groups  = vcat(fill(1, 3), fill(2, 3))
stacks  = vcat(1:3, 1:3)

fig = Figure(size = (600, 390))
ax = Axis(fig[1, 1]; xticks = (1:2, ["pedigree only", "pedigree + brood"]),
    ylabel = "variance (g²)", title = "where the same variation goes, in two models")
barplot!(ax, groups, heights; stack = stacks,
    color = Makie.wong_colors()[stacks])
Legend(fig[1, 2],
    [Makie.PolyElement(color = Makie.wong_colors()[k]) for k in 1:3], labels;
    framevisible = false)
fig

**Momo:** The two bars are almost the same height. The genetic block shrank and a new block appeared in its place.

**Itchy:** Almost the same height, because a variance partition is an accounting of one fixed quantity into named parts. But do not say "in its place": the cell measured where the brood block came from, and it is not where you are looking. Read the two shares.

**Momo:** Only `{julia} string(round(Int, 100 * (sigma_A_naive^2 - V_A) / V_B), "%")` of the brood block came out of the genetic block. `{julia} string(round(Int, 100 * (sigma_e_naive^2 - V_R) / V_B), "%")` of it came out of the **residual**.

**Itchy:** Most of it came out of the residual, and that is this morning's two-partitions argument finishing itself. The pedigree-only model has exactly one number for every full-sib pair — `{julia} round(0.5 * sigma_A_naive^2, digits = 3)`, half the additive variance — whether those two chicks shared a nest or not. The two-component model gives same-brood full sibs `{julia} round(0.5 * V_A + V_B, digits = 3)` and different-brood full sibs `{julia} round(0.5 * V_A, digits = 3)`. The first model had to *average over that split*, and what would not fit went into the residual, where it sat looking exactly like noise. So the mixed broods and spread pairs are not a curiosity about sparrow fieldwork — they are the whole reason the second model can be fitted at all.

**Itchy:** One more word about that denominator, since Objective 3 asks what is in it. **V_P here is conditional on the fixed effects.** Whatever Sex and Cohort explain is outside it — `{julia} round(var(fitted(full_model)), digits = 4)` g² on this file, so little that a full-variance denominator gives `{julia} round(V_A / (V_P + var(fitted(full_model))), digits = 4)` against `{julia} round(h2_full.estimate, digits = 4)`. Put in a covariate that explains a third of the variance and it will matter enormously, and nothing in the output will say so. Which brings us to the only question that matters.

### Which of those two numbers should you believe?

**Jaro:** Neither, until you have shown me the design can tell them apart.

**Itchy:** Neither, until we have shown the design can tell them apart, and there is exactly one way to show that and you have been doing it since Class 2. Compare them properly first, though: both were fitted by maximum likelihood on the same rows, so they are comparable.

In [ ]:
#| label: comparison
# A brood effect with NO pedigree at all: Class 6's model, on nests.
brood_only = drm(bf(@formula(Mass2 ~ Sex + Cohort + (1 | BroodNo))), Gaussian(); data = chicks)
fixed_only = drm(bf(@formula(Mass2 ~ Sex + Cohort)), Gaussian(); data = chicks)

@printf("%-26s %10s %10s %6s\n", "model", "logLik", "AIC", "dof")
for (lab, f) in (("fixed effects only", fixed_only), ("pedigree only (A)", naive),
                 ("brood only",         brood_only), ("pedigree + brood",  full_model))
    @printf("%-26s %10.2f %10.2f %6d\n", lab, loglik(f), aic(f), dof(f))
end
println()
@printf("AIC(A + brood) - AIC(pedigree only) = %.2f\n", aic(full_model) - aic(naive))
@printf("AIC(A + brood) - AIC(brood only)    = %.2f\n", aic(full_model) - aic(brood_only))

**Toto:** The model with both is the best of the four.

**Itchy:** By AIC, on these data, yes — and notice how little separates the last two rows. Before anybody reads a meaning into that gap, say what those two rows differ by. Eddie.

**Eddie:** One parameter. σ_A.

**Itchy:** One parameter, and it is **tested at zero**, which is the edge of the space a variance lives in. AIC's two-points-per-parameter penalty and the usual chi-squared reference are both worked out assuming the true value sits somewhere in the open middle of its range, and a variance at zero is on the edge. Class 8 taught the correction and Class 9 used it.

In [ ]:
#| label: boundary-test
# Dropping sigma_A tests a variance component at ZERO: the LR statistic follows a 50:50
# mixture of a point mass at zero and a chi-squared on 1 df, so the tail p-value is halved.
bt = lrt_boundary(full_model, brood_only; q = 1)

@printf("LR statistic for sigma_A = 0, given the brood : %.4f on %d df\n", bt.statistic, bt.q)
@printf("naive chi-squared p-value                     : %.4f\n", bt.pvalue_naive)
@printf("boundary-corrected p-value (Class 8's rule)   : %.4f\n", bt.pvalue)

**Toto:** The naive one is just the wrong side of a twentieth and the corrected one is comfortably on the other side.

**Itchy:** Which is the entire reason the correction exists, and why I will not let you read a `{julia} round(abs(aic(full_model) - aic(brood_only)), digits = 2)`-point AIC gap as a measurement. Notice what the corrected *p*-value does **not** buy you: it says the additive variance is not zero, not that we know what it is — the interval ran down to `{julia} round(h2_full.ci.lower, digits = 4)`, which is not a computed bound but the engine pushing a value that fell below zero back to the edge. Say so when you report it. Now the residuals, because a variance partition from a model that does not fit the data is a partition of nothing.

In [ ]:
#| label: fig-diagnostic
#| fig-cap: "Worm plot of the A-plus-brood model's quantile residuals, standardised by their own spread rather than by the residual sigma, because these residuals are marginal — they still carry the breeding values and the brood effects, as Class 6 explained — so the picture reads shape rather than scale. Both ends leave the band upward and the middle sags below it: right skew, from a mass with a hard floor at zero and no ceiling."
qr = residuals(full_model; type = :quantile)
fig_diagnostic(qr ./ std(qr); title = "chick mass, A + brood model")

**Momo:** Both ends leave the band upwards and the middle sags a little below it.

**Itchy:** Both ends up and the middle down is a ∪, and a ∪ in a worm plot has one name: the residuals are **skewed to the right**. A mass has a hard floor and no ceiling: a chick that weighs nothing is a chick that is not there, and a well-fed chick can be enormous. A right-skewed positive response is what a `Gamma` family is for; this book does not teach it, and on this file it would change the prediction intervals a great deal and the variance *ratio* very little. Not today. On to Jaro's question, which we answer the way this book always answers it: **make data where you know the truth, and see what the estimator says.**

**Itchy:** One thing first, and you may take it on trust. An animal-model fit takes the engine about two minutes on this file, and I want fifteen hundred of them. When every individual has one record there is a shortcut: rotate the data by the eigenvectors of *A*, and the model becomes a search over one number, the heritability. Because it is the likelihood written out, it can also do the one thing the engine would not, which is REML — that machinery, `profile_ll` and `fit_h2`, lives in `tools/diagnostics.jl`, documented there, because the linear algebra in it is not something this book teaches; the engine has no REML for a structured random effect at all ([Where the engine stops](appendix-b-engine.html)), which is the whole reason we are pricing the gap by hand. Here is the call and the two numbers it prints.

In [ ]:
#| label: sim-machinery
# With A = U diag(lambda) U', the covariance of U'y is diagonal: h2 * lambda_i + (1 - h2),
# times the total variance. `profile_ll` and `fit_h2` — the closed-form fixed-effect and
# variance fit at one h2, and the grid search over h2 that calls it — live in
# tools/diagnostics.jl, documented there; `recover` below reuses lambda and Xrot many times.
E = eigen(Symmetric(A))
lambda, U = E.values, E.vectors
Xdes = hcat(ones(n), Float64.(chicks.Sex .== "M"),
            [Float64(chicks.Year[i] == y) for i in 1:n, y in sort(unique(chicks.Year))[2:end]])
Xrot = U' * Xdes
grid = 0.0:0.001:0.995

yrot = U' * Float64.(chicks.Mass2)
ml  = fit_h2(yrot, lambda, Xrot, grid)
rml = fit_h2(yrot, lambda, Xrot, grid; reml = true)

@printf("h2 by ML   : the engine %.4f, this cell %.3f   (95%% profile interval %.3f to %.3f)\n",
        h2_naive.estimate, ml.h2, ml.lo, ml.hi)
@printf("h2 by REML : %.3f   (%+.1f%% of the ML value)\n", rml.h2, 100 * (rml.h2 / ml.h2 - 1))

**Momo:** The first line matches the engine. The second is the REML you said we could not have.

**Itchy:** Matching the engine is what earns this cell the right to stand in for it, and the second line is what REML would have reported: `{julia} round(rml.h2, digits = 3)` where ML says `{julia} round(ml.h2, digits = 3)`. Hold that number; the simulation will tell you whether the gap is the ML bias or an accident of one dataset. Three worlds, one estimator: **fit the animal-only model — the model Eddie was ready to publish — to data whose truth I chose.**

In [ ]:
#| label: recovery
L = cholesky(Symmetric(A)).L                       # a = sigma_A * L * z has covariance sigma_A^2 A
brood_index = let m = Dict(b => k for (k, b) in enumerate(brood_levels))
    [m[b] for b in chicks.BroodNo]
end
n_rep = 500

"Draw `n_rep` datasets with the stated truth and refit the ANIMAL-ONLY model to each."
function recover(sigma_a, sigma_b, sigma_r, rng)
    beta = Xdes \ Float64.(chicks.Mass2)      # any beta will do
    est = Float64[]; lo = Float64[]; hi = Float64[]
    for _ in 1:n_rep
        a = sigma_a .* (L * randn(rng, n))                 # breeding values
        b = sigma_b .* randn(rng, length(brood_levels))    # brood effects
        y = Xdes * beta .+ a .+ b[brood_index] .+ sigma_r .* randn(rng, n)
        f = fit_h2(U' * y, lambda, Xrot, grid)
        push!(est, f.h2); push!(lo, f.lo); push!(hi, f.hi)
    end
    return (est = est, lo = lo, hi = hi)
end

# (label, true sigma_A, true sigma_brood, true sigma_residual, generator)
worlds = [("genes only, no brood", sigma_A_naive, 0.0, sigma_e_naive, MersenneTwister(101)),
          ("genes and brood",      sA,            sB,  sE,            MersenneTwister(102)),
          ("brood only, no genes", 0.0,           sB,  sE,            MersenneTwister(103))]

results = [recover(w[2], w[3], w[4], w[5]) for w in worlds]
truths  = [w[2]^2 / (w[2]^2 + w[3]^2 + w[4]^2) for w in worlds]

@printf("%-22s %8s %8s %8s %9s %11s\n", "world", "true h2", "mean", "SD", "coverage", "excludes 0")
for (w, r, truth) in zip(worlds, results, truths)
    cover = count(i -> r.lo[i] <= truth <= r.hi[i], 1:n_rep) / n_rep
    excl  = count(>(0.0), r.lo) / n_rep
    @printf("%-22s %8.4f %8.4f %8.4f %9.3f %11.3f\n", w[1], truth, mean(r.est), std(r.est), cover, excl)
end

**Toto:** In the third world there are no genes at all and it says there are.

**Itchy:** In the third world every chick's breeding value is exactly zero, the only thing making siblings alike is the nest they grew up in, and the animal model — fitted to data with **no additive genetic variance whatsoever** — returns a mean heritability of `{julia} round(mean(results[3].est), digits = 3)`, which is `{julia} string(round(Int, 100 * mean(results[3].est) / h2_naive.estimate), "%")` of the number Eddie was ready to publish, and its interval excludes zero in `{julia} string(round(Int, 100 * count(>(0.0), results[3].lo) / n_rep), "%")` of the `{julia} n_rep` replicates. Out of a world with no genetics in it at all.

**Momo:** And the first world?

**Itchy:** The first world is the good news, with a bill attached. There the truth is `{julia} round(truths[1], digits = 3)` and the interval covers it in a fraction `{julia} round(count(i -> results[1].lo[i] <= truths[1] <= results[1].hi[i], 1:n_rep) / n_rep, digits = 3)` of the replicates, which is what a ninety-five per cent interval is supposed to do. But the mean estimate sits below the truth by `{julia} round(100 * abs(mean(results[1].est) - truths[1]) / truths[1], digits = 1)`% of it, and that is not noise: it is `{julia} round(abs(mean(results[1].est) - truths[1]) / (std(results[1].est) / sqrt(n_rep)), digits = 1)` Monte Carlo standard errors from zero. **That is the price of maximum likelihood on this design, measured**, and it is about the size of the REML-to-ML gap on the real sparrows, `{julia} round(100 * (rml.h2 / ml.h2 - 1), digits = 1)`%. So the sentence for a methods section is: *estimated by maximum likelihood, because REML is not available for a structured effect in this engine; a simulation from the fitted animal-only model puts ML's downward bias at about `{julia} round(100 * abs(mean(results[1].est) - truths[1]) / truths[1], digits = 1)`% of h².* Name the estimator, and price the limitation. Now draw all three.

In [ ]:
#| label: fig-recovery
#| fig-cap: "Histograms of estimated h² from the animal-only model, one per simulated world — no nest effect, a large nest effect, and no heritability at all — each with its truth marked by a dash-dot line in its colour, and the h² the real sparrows gave marked by the solid black line. World 1's truth is by construction the naive fit's own estimate, so its dash-dot line sits under the black one. The black line sits inside all three distributions: one variance ratio from one fit cannot tell the worlds apart."
# Histograms drawn by hand, so the three worlds share the same stated bins.
edges = range(0.0, 0.45, length = 46)
mids  = (edges[1:end-1] .+ edges[2:end]) ./ 2
count_in(v) = [count(x -> edges[b] <= x < edges[b + 1], v) for b in 1:length(mids)]

fig = Figure(size = (620, 400))
ax = Axis(fig[1, 1]; xlabel = "estimated h² from the animal-only model",
    ylabel = "replicates", title = "$(n_rep) simulated datasets per world, one estimator")
styles = [:solid, :dash, :dot]
for (k, (w, r, truth)) in enumerate(zip(worlds, results, truths))
    col = Makie.wong_colors()[k]
    lines!(ax, mids, Float64.(count_in(r.est)); color = col, linewidth = 2,
        linestyle = styles[k], label = w[1])
    if k == 1
        vlines!(ax, [truth]; color = col, linewidth = 1.5, linestyle = :dashdot, label = "truth")
    else
        vlines!(ax, [truth]; color = col, linewidth = 1.5, linestyle = :dashdot)
    end
end
vlines!(ax, [h2_naive.estimate]; color = :black, linewidth = 2.5, label = "real sparrows, h²")
Legend(fig[1, 2], ax; framevisible = false)
fig

**Itchy:** The dashed lines are the truths — the first world's sits underneath the solid black line, because that world was built from the real fit. The solid black line is the number we actually got from the sparrows. Momo, look at where it falls.

**Momo:** All three curves are over it. Every one of those worlds could have produced that number.

**Itchy:** Every one of them, and that is Jaro's question answered. The estimate from the real sparrows is what you would see if chick mass were `{julia} string(round(Int, 100 * truths[1]), "%")` heritable with no nest effect; if it were `{julia} string(round(Int, 100 * truths[2]), "%")` heritable with a large nest effect; and with **no heritability whatsoever**. **One variance ratio from one fit cannot distinguish those three worlds, and no amount of staring at the fit will make it.** What distinguishes them is a second variance component, and the model that has one says brood.

**Itchy:** So the honest report from this file is the second fit: **h² = `{julia} round(h2_full.estimate, digits = 3)`, 95% CI `{julia} round(h2_full.ci.lower, digits = 3)` to `{julia} round(h2_full.ci.upper, digits = 3)` with the lower bound floored at zero by the engine, estimated by maximum likelihood with a brood variance in the model**, and a sentence saying that without the brood term the same data give `{julia} round(h2_naive.estimate, digits = 3)`.

**Jaro:** Is there a design that does better, or is this the human condition?

**Itchy:** There is, and Kruuk and Hadfield name it in their abstract: the animal model separates genes from shared environment best *where pedigrees contain multiple generations*. Ours has **one**. Say why that matters, Eddie.

**Eddie:** A deep pedigree relates people who never shared a nest. Grandparents. Cousins. Anything two steps away.

**Itchy:** Anything two steps away carries relatedness with **no** common environment attached, and every such pair is a lever prying the two apart. The information lives in pairs, so count in pairs.

In [ ]:
#| label: leverage-pairs
brood = chicks.BroodNo
n_related = count(A[i, j] > 0 for i in 1:n for j in (i + 1):n)
n_split   = count(A[i, j] > 0 && brood[i] != brood[j] for i in 1:n for j in (i + 1):n)
@printf("related pairs: %d, of which %d (%.1f%%) never shared a brood\n",
        n_related, n_split, 100 * n_split / n_related)

**Itchy:** `{julia} round(100 * n_split / n_related, digits = 1)`% of the related pairs in this file never shared a nest, and that is why the two components could be separated here. So the rule is not "sib designs are hopeless". It is this: **the animal model separates genes from nest in proportion to how much of your relatedness comes from pairs who did not share one.** Count that in your own pedigree, in pairs, before you fit anything.

### The ceiling Class 6 gave you, and why it will not help today

**Eddie:** Class 6 said repeatability caps heritability. Use that. It is free.

**Itchy:** It is free and this file will not give it to me: `{julia} n` rows, `{julia} length(unique(chicks.ChickNo))` chicks, one weighing each. **A repeatability is the correlation between two measurements of the same individual, and there is no second measurement here, so R is not estimable from this file at all.** Nor will Class 6's wing repeatability do: the bound h² ≤ R is about one trait, measured twice, on the same individuals, and a grown sparrow's wing says nothing about a two-day-old chick's mass. Weigh the chicks twice and you would have an R for chick mass; it would cap today's h², and the *gap* between the two would be the permanent environment — which on this file already has a name: the brood. And remember Class 6's other warning: let maternal and early-life effects enter some measurements and not others, which is a fair description of a nest, and even the bound can fail (Dohm 2002).

**Eddie:** Class 6 promised twice that you would come back to the square root.

**Itchy:** So it did, and it is two sentences. **R caps h² directly, because both are ratios over the same total variance. R does not cap a *correlation*, because a correlation divides by a product of standard deviations rather than by a variance — so the ceiling there is the square root, √R,** a much looser cap that is routinely quoted as though it were the same number. Carry the right bound to the right problem.

### Four faces, one idea

**Itchy:** Last twenty minutes, and it is the reason this is one chapter rather than four. Everything we did today used **one** property of `A`: that it is a known matrix over the levels of a grouping factor, and a legal covariance. Nothing in the fit knew it was a pedigree.

| marker | what the matrix comes from | keyword | what the variance component means |
|---|---|---|---|
| `animal(1 \| id)` | a pedigree | `A =` | additive genetic variance |
| `relmat(1 \| id)` | anything you can compute | `K =` | whatever your matrix encodes |
| `phylo(1 \| species)` | a phylogeny | `tree =` | phylogenetic signal |
| `spatial(1 \| site)` | site coordinates | `coords =` | spatial variance, with a range |

**Itchy:** `relmat` is the general case. `phylo` and `spatial` are conveniences that *build* the matrix for you, from a tree or from coordinates; `animal` builds nothing today — it names the genetic interpretation and you still hand it `A`. Swap a pedigree for a **genomic kinship matrix** — the same relatedness numbers, read off shared DNA markers rather than guessed from a family tree — and you have written the animal model of the last fifteen years. Swap it for a phylogenetic correlation and "heritability" is called phylogenetic signal, and the arithmetic does not change. Swap it for `exp(-d/ρ)` over site coordinates and nearby sites are correlated — and there the model estimates the range ρ as well, because nobody hands you the correlation.

**Toto:** Can we fit one?

**Itchy:** Not today, and the reason is the whole ethic of this book. **There is no phylogeny in this repository and there are no site coordinates.** I could invent a tree in four lines and show you a fit, and the number would be a number about my four lines. The call sites are in the table, the tutorials are in the further reading, and the first time you fit one it should be on your own tree.

**Eddie:** And phylogenetic comparative methods proper? Meta-analysis?

**Itchy:** Both are today's problem — non-independence from a shared history, or from shared effect sizes — and both need a book rather than an afternoon. They are not in this one, deliberately.

**Momo:** And nobody builds that matrix by hand in practice.

**Itchy:** In practice a package usually builds it for you — the R twin of this engine will walk a three-column pedigree of `id`, `dam` and `sire` into `A` itself — which is why this chapter spent a page doing it in the open, because a matrix a package built for you is one you should still check, three entries by name, exactly as above.

**Toto:** So today's summary is: tell the model who is related to whom.

**Itchy:** Tell the model who is related to whom, and then be extremely careful about what *else* is true of the people you just told it about. The matrix is a hypothesis. Test it like one.

---

## Summary

### Stats stuff

- **The last constant, freed.** Classes 6, 7 and 9 let groups differ but drew every group's effect independently, `u ~ N(0, σ_b² I)`. A structured random effect replaces the `I` with a **known** matrix: `a ~ N(0, σ_A² A)`. You supply the matrix; the model estimates one scalar, how much that structure is worth.
- **The additive relationship matrix.** *A*ᵢⱼ = ½(*A*ᵢ,ₛᵢᵣₑ₍ⱼ₎ + *A*ᵢ,dam₍ⱼ₎), *A*ᵢᵢ = 1 + *F*ᵢ. With unrelated non-inbred founders and one generation this collapses to ¼ per shared parent: full sibs ½, half sibs ¼. Build it in a cell and check named entries against the file — a matrix can be the right *shape* and still be indexed wrongly.
- **The animal model.** `y = Xβ + a + ε`, `a ~ N(0, σ_A² A)`. **h² = V_A / V_P** is a ratio of variance components, exactly like Class 6's repeatability, and V_P is **conditional on the fixed effects**: whatever your covariates explain is outside it. Negligible on this file, decisive on one where the covariates matter.
- **The confound that eats heritabilities.** A relatedness matrix says relatives share genes, not that they shared a nest, a mother's condition, a territory or a year. Anything else that makes relatives alike, and is not in the model, is absorbed by σ_A and reported as heritability (Kruuk & Hadfield 2007). Fitting the brood here cut h² by about `{julia} string(round(Int, 100 * (1 - h2_full.estimate / h2_naive.estimate)), "%")`, and `{julia} string(round(Int, 100 * (sigma_e_naive^2 - V_R) / V_B), "%")` of the brood variance came out of the **residual**, where the difference between same-brood and different-brood full sibs had been hiding.
- **Whether you can separate them is a property of the design, not of the software.** If every set of full sibs is one brood, "same parents" and "same nest" are the same partition and no modelling will split them. Cross-fostering, or a **deep** pedigree in which relatives two steps apart never shared an environment, is what makes the two components estimable. Count the related pairs that never shared a group in your own file before you fit.
- **A simulation answers the question a fit cannot.** Data simulated with *no* additive genetic variance, in which siblings resemble each other only because they shared a nest, gave a mean heritability of `{julia} round(mean(results[3].est), digits = 3)` from the animal-only model, with an interval excluding zero almost every time. The estimator is not broken; the design is confounded, and only a simulation with a known truth tells you which of those you are looking at.
- **The Class 6 ceiling needs the same trait, twice.** h² ≤ R holds for one trait measured repeatedly on the same individuals; this file has one weighing per chick, so R is not estimable here, and a repeatability of another trait caps nothing. A **correlation** with a trait of repeatability R is capped by √R, not by R. The gap between R and h² is the permanent environment — here, the brood. Let maternal and early-life effects enter some measurements and not others, and even the bound can fail (Dohm 2002).
- **Maximum likelihood, because REML is not on offer here — and its price, measured.** A simulation from the fitted animal-only model puts ML's downward bias on h² at about `{julia} round(100 * abs(mean(results[1].est) - truths[1]) / truths[1], digits = 1)`%; REML computed from the same likelihood by hand moves the real-data estimate from `{julia} round(ml.h2, digits = 3)` to `{julia} round(rml.h2, digits = 3)`. Name the estimator, and price the limitation.
- **A variance component tested at zero is a boundary test.** Dropping σ_A from the two-component model gives a likelihood-ratio statistic of `{julia} round(bt.statistic, digits = 2)` on 1 degree of freedom: *p* = `{julia} round(bt.pvalue_naive, digits = 4)` against an ordinary chi-squared table, *p* = `{julia} round(bt.pvalue, digits = 3)` against the halved reference Class 8 taught (Self & Liang 1987; Stram & Lee 1994). The reading flips on the correction.
- **A ratio of estimates is not the estimate of a ratio.** `heritability` returns `estimate`, `bias` and `corrected`; on the two-component fit the correction is `{julia} string(round(Int, 100 * h2_full.bias / h2_full.estimate), "%")` of the headline number, as Class 6 forecast. Report `corrected` and `bias`. A bound of exactly 0 or 1 is the engine pushing a value that fell outside back to the edge; say so when you report it.
- **Four faces, one model.** `animal` (pedigree), `relmat` (any matrix), `phylo` (a tree), `spatial` (coordinates): four sources for one matrix. `spatial` additionally estimates a range, because nobody hands you the correlation between two places.

### Julia you used

- **`[expr for i in 1:n, j in 1:n]`.** A comprehension with two `for`s in one bracket builds a **matrix**. `[A[i, j] for i in 1:n for j in (i + 1):n]`, with the second `for` depending on the first, flattens into one vector.
- **`0.25 * (sire[i] == sire[j])`.** A comparison is true or false, and arithmetic treats true as one and false as zero. `(a == b) + (c == d) == 1` counts how many of two things hold.
- **`A'`.** The transpose. `A == A'` is the symmetry check.
- **`findfirst(i -> any(j -> f(i, j), 1:n), 1:n)`.** `any` asks whether a function is true for some element; `findfirst` returns the first position where its function is true.
- **`v[1:min(end, 18)]`.** `end` inside brackets means the last position.
- **`groupby(df, [:Dad, :Mum])`.** Grouping by two columns at once.
- **`Matrix{Float64}(I, n, n)`, `vcat`, `fill(1, 3)`.** An identity matrix, vectors joined end to end, and a vector of one repeated value.
- **`let m = Dict(...) ... end`.** A block with a temporary name that does not leak out.
- **`if k == 1 ... else ... end`.** The full form of `if`, for when the branches are statements rather than values.
- **`cholesky(Symmetric(A)).L`.** How you *draw* a correlated random effect: `σ_A * L * randn(rng, n)` has covariance σ_A² A. Beyond it, the linear algebra behind the REML fit — `eigen`, `\`, `hcat` — is not taught here; `profile_ll` and `fit_h2` do it in `tools/diagnostics.jl`.

### Calls you used

- `animal(1 | id)` in the mean formula, with `A = A` as a keyword to `drm`: the animal model. `relmat(1 | id)` with `K = K` is the same term with a matrix you name yourself; `algorithm = :sparse` assembles the same fit in seconds rather than minutes.
- **The matrix is indexed by the levels of the grouping factor in the order they first appear in the data.** Build it from the same column you group on, in one cell, and it cannot drift.
- `heritability(fit)` and `repeatability(fit)`: the two ratios, each returning `estimate`, `bias`, `corrected`, `se` and `ci`. With more than one structured component, pass `component = :name`.
- `re_sd(fit)[:id]`, `sigma(fit)`, `is_converged(fit)`, `dof(fit)`: the component SDs, the residual SD, whether the optimiser finished, and the parameter count.
- `lrt_boundary(full, reduced; q = 1)`: the boundary-corrected test for a variance component at zero. Both fits must be ML on the same rows.
- `residuals(fit; type = :quantile)`: Class 5a's quantile residuals, available on a structured fit like any other.
- **The simulation thread.** Every chapter ends by asking the fit to invent data; here the cell is a reminder.

In [ ]:
#| label: simulate-refit
rng_sim = MersenneTwister(20261007)     # a seed is a promise: the same draw every time the site is built
ysim = simulate(full_model; nsim = 50, rng = rng_sim)
dev = ysim .- fitted(full_model)

@printf("observed SD about the fixed part  : %.4f\n", std(chicks.Mass2 .- fitted(full_model)))
@printf("simulated SD about the fixed part : %.4f\n", mean(std(dev[:, k]) for k in 1:size(dev, 2)))
@printf("residual sigma alone              : %.4f\n", sE)
@printf("sqrt(V_A + V_B + V_R)             : %.4f\n", sqrt(V_P))

The simulated spread is the residual σ, not the phenotypic total: `simulate` draws with every random effect at zero, as Class 6 found, which is why the recovery study drew the breeding values itself.

---

## Further reading

*Graded by depth. Details checked on 2026-09-07 against OpenAlex, an open catalogue of papers.*

1. **Wilson, A. J., Réale, D., Clements, M. N., Morrissey, M. M., Postma, E., Walling, C. A., Kruuk, L. E. B. & Nussey, D. H. (2010) "An ecologist's guide to the animal model", *Journal of Animal Ecology* 79:13–26.** doi:10.1111/j.1365-2656.2009.01639.x (online 2009). The one to read first: three worked tutorials for ecologists, ending in a list of the pitfalls.
2. **Kruuk, L. E. B. (2004) "Estimating genetic parameters in natural populations using the 'animal model'", *Philosophical Transactions of the Royal Society B* 359:873–890.** doi:10.1098/rstb.2003.1437. The paper that made animal models standard in wild-population ecology, and the one that describes the literature as **restricted** maximum-likelihood animal models — which is why this chapter had to price the ML fit it was given.
3. **Kruuk, L. E. B. & Hadfield, J. D. (2007) "How to separate genetic and environmental causes of similarity between relatives", *Journal of Evolutionary Biology* 20:1890–1903.** doi:10.1111/j.1420-9101.2007.01377.x. The whole paper is today's confound. Read it before you report a heritability from a design where full sibs share a nest, and read it twice if your design has no cross-fostering in it.
4. **Dohm, M. R. (2002) "Repeatability estimates do not always set an upper limit to heritability", *Functional Ecology* 16:273–280.** doi:10.1046/j.1365-2435.2002.00621.x. Class 6's counterweight: its abstract lists the conditions under which the free bound fails, and the one that matters on this page is **maternal effects** — in this chapter's data, precisely the brood.
5. **Lynch, M. & Walsh, B. (1998) *Genetics and Analysis of Quantitative Traits*. Sinauer.** The reference work: chapters 7 and 26–27 for the relationship matrix and the mixed-model machinery. A thousand pages you look things up in for the rest of your career.
6. **DRM.jl's tutorials `animal-models.md`, `relmat-known-matrices.md`, `phylogenetic-models.md` and `spatial-models.md`.** The four faces, each with a status note saying which families and structures are implemented today. Read the status note before you plan an analysis around a call.
7. **Self, S. G. & Liang, K.-Y. (1987) "Asymptotic properties of maximum likelihood estimators and likelihood ratio tests under nonstandard conditions", *Journal of the American Statistical Association* 82(398):605–610.** doi:10.1080/01621459.1987.10478472. **Stram, D. O. & Lee, J. W. (1994) "Variance components testing in the longitudinal mixed effects model", *Biometrics* 50(4):1171.** doi:10.2307/2533455. **Patterson, H. D. & Thompson, R. (1971) "Recovery of inter-block information when block sizes are unequal", *Biometrika* 58(3):545–554.** doi:10.1093/biomet/58.3.545. Self and Liang for the halved *p*-value behind `lrt_boundary`, Stram and Lee for the same rule in a mixed model, and Patterson and Thompson for REML itself — the two lines the simulation cell adds by hand.

---

## Exercises

In [ ]:
#| label: exercise-checks
#| echo: false
#| output: false
# Numbers the exercise checks quote, computed from the named file and the chapter's own objects.
ex_td = CSV.read("data/ch8/tadpoles.csv", DataFrame)
ex_pond = ex_td.pond
ex_nt = nrow(ex_td)
ex_K = [i == j ? 1.0 : 1.0 * (ex_pond[i] == ex_pond[j]) for i in 1:ex_nt, j in 1:ex_nt]
ex_off = [ex_K[i, j] for i in 1:ex_nt for j in (i + 1):ex_nt]
ex_same_pond = count(==(1.0), ex_off)
ex_min_eig = minimum(eigvals(Symmetric(ex_K)))
ex_full_split = count(A[i, j] == 0.5 && brood[i] != brood[j] for i in 1:n for j in (i + 1):n)
ex_half_split = count(A[i, j] == 0.25 && brood[i] != brood[j] for i in 1:n for j in (i + 1):n)
ex_third = recover(0.0, sB / 2, sE, MersenneTwister(104))
ex_third_mean = mean(ex_third.est)
ex_third_excl = count(>(0.0), ex_third.lo) / n_rep

Graded by depth: the first two take ten minutes each. Every exercise names a file under `data/` that exists, or uses the objects this chapter built; do it on your own organism as well where you have one. A *check* is a number computed when this page was built — match it before going on. **For every question, paste your code and then explain in your own words what each line does**, as if to somebody who has done Class 6 and not Class 10.

1. **Build a matrix and check it.** Momo's tadpoles in `data/ch8/tadpoles.csv` have no pedigree, so invent the rule a clonal design would give: every tadpole in a pond is treated as genetically identical to its pond-mates, so their entry is 1, and everyone else's is 0. Build that matrix with a two-`for` comprehension as the chapter built `A`, print its distinct off-diagonal values and count the pairs that share a pond, check three entries by name against the file, and print its smallest eigenvalue. Then two sentences: why the eigenvalue check fails on this matrix and had to, and why the ratio a fit to it would return is a **broad**-sense heritability and not the narrow-sense one this chapter estimated. *Check:* off-diagonal values `{julia} join(sort(unique(ex_off)), " and ")`; `{julia} ex_same_pond` of the `{julia} length(ex_off)` pairs share a pond; the smallest eigenvalue rounds to `{julia} abs(round(ex_min_eig, digits = 6))`, so `isposdef` says false — a block of ones has rank one, and this matrix is `(1 | pond)` from Class 6 wearing a different name.

2. **Find the confound, in pairs.** The chapter counted the related pairs that never shared a nest. Split that count: from `A` and `chicks.BroodNo`, count the **full**-sib pairs (entry ½) in different broods and the **half**-sib pairs (entry ¼) in different broods, with the chapter's double-`for` comprehension. Then two sentences: which of the two kinds of pair does the separating of genes from nest, and what a field season that produced none of them would leave you unable to estimate. *Check:* `{julia} ex_full_split` full-sib pairs and `{julia} ex_half_split` half-sib pairs never shared a brood.

3. **The third world, with the confound halved.** The chapter's third world had no genetic variance and the fitted brood SD. Call the chapter's `recover` with the genetic SD at exactly zero, the brood SD at **half** the fitted `sB`, the residual SD at `sE`, and `MersenneTwister(104)`. Report the mean estimated h² and the proportion of replicates whose interval excludes zero, and state the number of replicates. Then answer: did halving the confound halve the phantom heritability, and what does the answer tell you about how the nest variance is being counted? *Check:* mean h² `{julia} round(ex_third_mean, digits = 3)` over `{julia} n_rep` replicates, interval excluding zero in a proportion `{julia} round(ex_third_excl, digits = 2)` of them — against the chapter's third world at `{julia} round(mean(results[3].est), digits = 3)`.

4. **Order matters.** Take the chapter's `A`, shuffle its rows and columns with one permutation, `perm = randperm(MersenneTwister(1), n)` and `A[perm, perm]`, and refit the animal-only model with the shuffled matrix and the data untouched — this takes the engine several minutes. Report `is_converged`, σ_A and h². Then write one sentence explaining why nothing in the output warned you. *Check:* the check is what is missing — `is_converged` says true, no warning appears, and the printout looks like any other fit. Whatever σ_A came out, nothing told you that every chick had just been handed a stranger's relatives.